# Ensemble Affinity Analysis (On-the-fly)

This notebook computes PRP and RA summaries in memory from merged benchmark files.

No summary CSVs are created and figures are shown inline only.

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px

In [ ]:
# Set run_id explicitly, or keep None to use latest run folder.
# run_id =None
# run_id = "20260401_082538"
# run_id = "20260406_211752"

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
bench_root = repo_root / "data" / "results" / "affinity_benchmark"

if run_id is None:
    run_dirs = sorted([p for p in bench_root.iterdir() if p.is_dir()])
    if not run_dirs:
        raise FileNotFoundError(f"No benchmark runs found under {bench_root}")
    run_dir = run_dirs[-1]
else:
    run_dir = bench_root / run_id

prp_path = run_dir / "merged_prp_affinities.csv"
ra_path = run_dir / "merged_ra_affinities.csv"

if not prp_path.exists() or not ra_path.exists():
    raise FileNotFoundError(
        "Merged files not found. Run scripts/run_postprocessing.py first for this run_id."
    )

prp_df = pd.read_csv(prp_path)
ra_df = pd.read_csv(ra_path)

ignore_models = ['smollm2-1.7b', 'gemma3-4b', 'phi3-3.8b']
prp_df = prp_df[~prp_df['Model_Slug'].isin(ignore_models)]
ra_df = ra_df[~ra_df['Model_Slug'].isin(ignore_models)]
print("Run directory:", run_dir)
print("PRP rows:", len(prp_df), "| Models:", prp_df['Model_Slug'].nunique())
print("RA rows:", len(ra_df), "| Models:", ra_df['Model_Slug'].nunique())

FileNotFoundError: Merged files not found. Run scripts/run_postprocessing.py first for this run_id.

In [ ]:
# Quick coverage checks
display(prp_df[['Model_Slug', 'Abstract_Index']].drop_duplicates().groupby('Model_Slug').size().rename('n_abstracts').reset_index())
display(ra_df[['Model_Slug', 'Abstract_Index']].drop_duplicates().groupby('Model_Slug').size().rename('n_abstracts').reset_index())

# PRP Affinity level
Affinity level of an **abstract** to each **primary research programme** (PRP)

In [ ]:
fig = px.box(
    prp_df,
    x='Abstract_Index',
    y='LLM_Affinity',
    points='all',
    color = 'PRP_Name',
    title=f'PRP Affinity Distribution by Abstract'
 )
fig.update_xaxes(range=[5.5,11.5])
fig.update_layout(xaxis_title='Abstract Index', yaxis_title='LLM Affinity')
fig.show()

NameError: name 'prp_df' is not defined

In [ ]:
fig = px.box(
    prp_df,
    color='Model_Slug',
    y='LLM_Affinity',
    x = 'PRP_Name',
    # points = 'all',
    title='Affinity Distribution by PRP across models',
 )
# fig.update_layout(xaxis_title='PRP', yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
# fig = px.box(
#     prp_df,
#     x='Abstract_Index',
#     y='LLM_Affinity',
#     points='all',
#     color = 'PRP_Name',
#     title=f'PRP Affinity Distribution by Abstract'
#  )
# fig.update_xaxes(range=[0.5,10.5])
# fig.update_layout(xaxis_title='Abstract Index', yaxis_title='LLM Affinity')
# fig.show()

In [5]:
abstracts_ids = [1,8,20, 19]
abstracts_ids = [20, 25, 34]
fig = px.box(
    prp_df[prp_df['Abstract_Index'].isin(abstracts_ids)], #ra_df[ra_df['Abstract_Index'].isin([7,8,9])],
    # x='Abstract_Index',
    y='LLM_Affinity',
    # color='Model_Slug',
    points='all',
    # hover_data='PRP_Name',
    # facet_row='PRP_Name',
    facet_col='Abstract_Index',
    color = 'PRP_Name',
    # x = 'Model_Slug',
    title=f'PRP Affinity Distribution by Abstract'
 )
# fig.update_xaxes(range=[0.5,10.5])
fig.update_layout(xaxis_title='Abstract Index', yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig.show()

NameError: name 'prp_df' is not defined

In [ ]:
fig = px.line(
    prp_df[prp_df['Abstract_Index'].isin([1,8,20, 19, 30])], #ra_df[ra_df['Abstract_Index'].isin([7,8,9])],
    # x='Abstract_Index',
    y='LLM_Affinity',
    # color='Model_Slug',
    # points='all',
    # hover_data='PRP_Name',
    # facet_row='PRP_Name',

    facet_col='Abstract_Index',
    color = 'PRP_Name',
    x = 'Model_Slug',
    title=f'PRP Affinity Distribution by Abstract',
    markers = True
 )
# fig.update_xaxes(range=[0.5,10.5])
fig.update_layout(yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig.show()

General categories:
- 1 - Scenario Reduction Network Based on Wasserstein Distance With Regularization - Planning
- 8 - Simultaneous Assessment of Multiple Aspects of Stability of Power Systems With Renewable Generation -  Stability
- 19 - Chance-Constrained Pre-Contingency Joint Self- Scheduling of Energy and Reserve in VPP - DER, Services
- 20 - Dual Grid-Forming Control With Energy Regulation Capability of MMC-HVDC System Integrating Offshore Wind Farms and Weak Grids - IBR, Stability
- 30 - Synergistic Integration of Machine Learning and Mathematical Optimization for Unit Commitment - ????

Observations:
- Gemma: fails to identify planning for Index 1
- Gemma cloud: best classification
- Llama: miscelaneous performance
- Mistral: systematically assign too high affinities
- Phi & Smollm: systematically assign too high affinities
- Qwen: miscelaneous


## PRP Summaries

In [ ]:
prp_summary = (
    prp_df.groupby(['Model_Slug', 'PRP_Name'], as_index=False)['LLM_Affinity']
    .agg(mean_affinity='mean', std_affinity='std', n='count')
)
display(prp_summary.head(20))

In [ ]:
# 
fig = px.scatter(
    prp_summary,
    color='PRP_Name',
    y='mean_affinity',
    x = 'Model_Slug',
    title='PRP Affinity Distribution by Model',
    # points = 'all',
 )
fig.update_layout(xaxis_title='PRP', yaxis_title='LLM Affinity')
fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
prp_heat = prp_summary.pivot(index='Model_Slug', columns='PRP_Name', values='mean_affinity').sort_index()
fig = px.imshow(
    prp_heat,
    text_auto='.1f',
    color_continuous_scale='Viridis',
    title='Mean PRP Affinity (Model x PRP)',
    aspect='auto'
 )
fig.update_layout(xaxis_title='PRP Name', yaxis_title='Model')
fig.show()

In [ ]:
# Top-1 PRP agreement per abstract
prp_top1 = (
    prp_df.sort_values(['Model_Slug', 'Abstract_Index', 'LLM_Affinity'], ascending=[True, True, False])
    .groupby(['Model_Slug', 'Abstract_Index'], as_index=False)
    .first()[['Model_Slug', 'Abstract_Index', 'PRP_Name','LLM_Affinity']]
)

prp_agreement = (
    prp_top1.groupby('Abstract_Index')['PRP_Name']
    .nunique()
    .rename('n_unique_top1_prp')
    .reset_index()
)

print('Full agreement rate (all models same top-1 PRP):', (prp_agreement['n_unique_top1_prp'] == 1).mean())
display(prp_agreement['n_unique_top1_prp'].value_counts().sort_index().rename('n_abstracts'))

## RA Summaries

In [ ]:
ra_df.head(20)

In [ ]:
# fig = px.box(
#     ra_df,
#     x='Abstract_Index',
#     y='LLM_Affinity',
#     color='Model_Slug',
#     points='all',
#     # facet_row='PRP_Name',
#     # color = 'PRP_Name',
#     title=f'RA Affinity Distribution by Abstract'
#  )
# fig.update_layout(xaxis_title='Abstract Index', yaxis_title='LLM Affinity')
# # fig.update_xaxes(tickangle=45)
# fig.show()

In [ ]:
fig = px.scatter(#prp_df[prp_df['Abstract_Index'].isin([1,8,20,19])]
    ra_df[ra_df['Abstract_Index'].isin([1,8,20,19])],
    # ra_df,
    x='RA2025_ID',
    y='LLM_Affinity',
    color='Model_Slug',
    facet_col = 'Abstract_Index',
    # points='all',
    hover_data = 'Model_Slug',
    # facet_row='PRP_Name',
    # color = 'PRP_Name',
    # title=f'RA Affinity Distribution by Abstract'
 )
fig.update_layout(xaxis_title='RA question', yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
ra_summary = (
    ra_df.groupby(['Model_Slug', 'RA2025_ID'], as_index=False)['LLM_Affinity']
    .agg(mean_affinity='mean', std_affinity='std', n='count')
)
display(ra_summary.head(20))

In [ ]:
fig = px.box(
    prp_df,
    x='Model_Slug',
    y='LLM_Affinity',
    title='RA Affinity Distribution by Model'
 )
fig.update_layout(xaxis_title='Model', yaxis_title='LLM Affinity')
fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
fig = px.box(
    ra_df,
    x='Model_Slug',
    y='LLM_Affinity',
    title='RA Affinity Distribution by Model'
 )
fig.update_layout(xaxis_title='Model', yaxis_title='LLM Affinity')
fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
# Top-1 RA agreement per abstract
ra_top1 = (
    ra_df.sort_values(['Model_Slug', 'Abstract_Index', 'LLM_Affinity'], ascending=[True, True, False])
    .groupby(['Model_Slug', 'Abstract_Index'], as_index=False)
    .first()[['Model_Slug', 'Abstract_Index', 'RA2025_ID']]
)

ra_agreement = (
    ra_top1.groupby('Abstract_Index')['RA2025_ID']
    .nunique()
    .rename('n_unique_top1_ra')
    .reset_index()
)

print('Full agreement rate (all models same top-1 RA):', (ra_agreement['n_unique_top1_ra'] == 1).mean())
display(ra_agreement['n_unique_top1_ra'].value_counts().sort_index().rename('n_abstracts'))

## Coherency Analysis: Model Agreement on Affinity Levels

Analyze whether all models assign similar affinity scores to the same questions/abstracts.

## Coherency Logic: Coefficient of Variation (CV)

The coefficient of variation measures how dispersed the model affinities are relative to their average value.  
For each abstract-target unit, we collect the affinity scores from all models and compute:

$$
\mu = \frac{1}{n}\sum_{i=1}^{n} x_i
$$

$$
\sigma = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(x_i-\mu)^2}
$$

$$
CV = \frac{\sigma}{\mu + \varepsilon}
$$

where:

- $x_i$ is the affinity score assigned by model $i$
- $n$ is the number of models
- $\mu$ is the mean affinity
- $\sigma$ is the sample standard deviation
- $\varepsilon$ is a small constant added to avoid division by zero

Interpretation:

- Low CV means the models are in close agreement
- High CV means the models disagree more strongly
- CV can become unstable when the mean affinity is very small, so low-mean cases should be interpreted with caution

In [ ]:
def affinity_level_fixed(score):
    if pd.isna(score):
        return pd.NA
    try:
        x = float(score)
    except Exception:
        return pd.NA
    if 0 <= x <= 40:
        return 'Low'
    if 40 < x <= 70:
        return 'Moderate'
    if 70 < x <= 100:
        return 'High'
    return pd.NA


def compute_unit_coherency(df, target_col, eps=1e-8, low_mean_threshold=5.0):
    required_cols = {'Abstract_Index', 'Model_Slug', target_col, 'LLM_Affinity'}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    unit_stats = (
        df.groupby(['Abstract_Index', target_col], as_index=False)['LLM_Affinity']
        .agg(
            mean_affinity='mean',
            std_affinity='std',
            min_affinity='min',
            max_affinity='max',
            n_models='count',
            mad_affinity=lambda s: float(np.mean(np.abs(s - s.mean()))),
        )
    )
    unit_stats['cv'] = unit_stats['std_affinity'] / (unit_stats['mean_affinity'] + eps)
    unit_stats.loc[unit_stats['n_models'] < 2, 'cv'] = np.nan
    unit_stats['range'] = unit_stats['max_affinity'] - unit_stats['min_affinity']
    unit_stats['Affinity_Level'] = unit_stats['mean_affinity'].apply(affinity_level_fixed)
    unit_stats['cv_low_mean_flag'] = unit_stats['mean_affinity'] < float(low_mean_threshold)
    unit_stats['singleton_flag'] = unit_stats['n_models'] < 2
    return unit_stats


def summarize_coherency_by_level(unit_stats):
    level_order = pd.CategoricalDtype(categories=['Low', 'Moderate', 'High'], ordered=True)
    out = unit_stats.copy()
    out['Affinity_Level'] = out['Affinity_Level'].astype(level_order)

    level_summary = (
        out.dropna(subset=['Affinity_Level'])
        .groupby('Affinity_Level', as_index=False, observed=False)
        .agg(
            n_units=('Affinity_Level', 'count'),
            mean_cv=('cv', 'mean'),
            median_cv=('cv', 'median'),
            p90_cv=('cv', lambda s: float(np.percentile(s.dropna(), 90)) if s.notna().any() else np.nan),
            mean_mad=('mad_affinity', 'mean'),
            median_mad=('mad_affinity', 'median'),
            mean_range=('range', 'mean'),
            pct_low_mean_flag=('cv_low_mean_flag', 'mean'),
            pct_singleton=('singleton_flag', 'mean'),
        )
        .sort_values('Affinity_Level')
    )
    level_summary['pct_low_mean_flag'] = 100.0 * level_summary['pct_low_mean_flag']
    level_summary['pct_singleton'] = 100.0 * level_summary['pct_singleton']
    return level_summary


def compute_pairwise_disagreement_by_level(df, target_col):
    records = []
    for (abstract_idx, target_id), grp in df.groupby(['Abstract_Index', target_col]):
        grp2 = grp[['Model_Slug', 'LLM_Affinity']].dropna()
        if grp2['Model_Slug'].nunique() < 2:
            continue

        level = affinity_level_fixed(grp2['LLM_Affinity'].mean())
        rows = grp2.sort_values('Model_Slug')[['Model_Slug', 'LLM_Affinity']].to_records(index=False)

        for i in range(len(rows)):
            for j in range(i + 1, len(rows)):
                m1, a1 = rows[i]
                m2, a2 = rows[j]
                records.append(
                    {
                        'Abstract_Index': abstract_idx,
                        'Target_ID': target_id,
                        'Affinity_Level': level,
                        'Model_1': str(m1),
                        'Model_2': str(m2),
                        'Affinity_Difference': abs(float(a1) - float(a2)),
                    }
                )

    pairwise = pd.DataFrame(records)
    if pairwise.empty:
        return pairwise, pairwise

    pairwise_summary = (
        pairwise.dropna(subset=['Affinity_Level'])
        .groupby(['Affinity_Level', 'Model_1', 'Model_2'], as_index=False)
        .agg(mean_diff=('Affinity_Difference', 'mean'), median_diff=('Affinity_Difference', 'median'), std_diff=('Affinity_Difference', 'std'), n_pairs=('Affinity_Difference', 'count'))
        .sort_values(['Affinity_Level', 'mean_diff'], ascending=[True, False])
    )
    return pairwise, pairwise_summary

In [ ]:
prp_coherency = compute_unit_coherency(prp_df, target_col='PRP_Name')

print("PRP Coherency Summary:")
print(f"Mean CV across all abstract-PRP pairs: {prp_coherency['cv'].mean():.3f}")
print(f"Abstracts with perfect agreement (CV=0): {(prp_coherency['cv'] == 0).sum()}")
print("\nBottom 10 coherent (high CV) abstract-PRP pairs:")
display(prp_coherency.nlargest(10, 'cv')[['Abstract_Index', 'PRP_Name', 'mean_affinity', 'cv', 'range']])
print("\nTop 10 coherent (low CV) abstract-PRP pairs:")
display(prp_coherency.nsmallest(10, 'cv')[['Abstract_Index', 'PRP_Name', 'mean_affinity', 'cv', 'range']])

In [ ]:
prp_coherency

In [ ]:
fig1 = px.box(
    prp_df[prp_df['Abstract_Index'].isin(prp_coherency.nlargest(7, 'cv')['Abstract_Index'])],
    y='LLM_Affinity',
    points='all',
    facet_col='Abstract_Index',
    color = 'PRP_Name',
    title=f'PRP Affinity Distribution (top CV abstracts)'
 )
fig2 = px.box(
    prp_df[prp_df['Abstract_Index'].isin(prp_coherency.nsmallest(5, 'cv')['Abstract_Index'])],
    y='LLM_Affinity',
    points='all',
    facet_col='Abstract_Index',
    color = 'PRP_Name',
    title=f'PRP Affinity Distribution (low CV abstracts)'
 )
# fig.update_xaxes(range=[0.5,10.5])
fig1.update_layout(yaxis_title='LLM Affinity')
fig2.update_layout(yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig1.show()
fig2.show()

In [ ]:
ra_coherency = compute_unit_coherency(ra_df, target_col='RA2025_ID')

print("RA Coherency Summary:")
print(f"Mean CV across all abstract-RA pairs: {ra_coherency['cv'].mean():.3f}")
print(f"Abstracts with perfect agreement (CV=0): {(ra_coherency['cv'] == 0).sum()}")
print("\nBottom 10 coherent (high CV) abstract-RA pairs:")
display(ra_coherency.nlargest(10, 'cv')[['Abstract_Index', 'RA2025_ID', 'mean_affinity', 'cv', 'range']])
print("\nTop 10 coherent (low CV) abstract-RA pairs:")
display(ra_coherency.nsmallest(10, 'cv')[['Abstract_Index', 'RA2025_ID', 'mean_affinity', 'cv', 'range']])

In [ ]:
# Abstract-level coherency: aggregate CV per abstract (for both PRP and RA)
prp_abstract_coherency = (
    prp_coherency.groupby('Abstract_Index', as_index=False)['cv']
    .agg(mean_cv='mean', max_cv='max', min_cv='min', std_cv='std', n_pairs='count')
    .sort_values('mean_cv', ascending=True)
)

ra_abstract_coherency = (
    ra_coherency.groupby('Abstract_Index', as_index=False)['cv']
    .agg(mean_cv='mean', max_cv='max', min_cv='min', std_cv='std', n_pairs='count')
    .sort_values('mean_cv', ascending=True)
)

print("PRP Abstract-level Coherency (Average CV per Abstract):")
print(f"Most coherent abstracts (lowest mean CV):")
display(prp_abstract_coherency.head(10))
print(f"\nLeast coherent abstracts (highest mean CV):")
display(prp_abstract_coherency.tail(10))

print("\n" + "="*60)
print("\nRA Abstract-level Coherency (Average CV per Abstract):")
print(f"Most coherent abstracts (lowest mean CV):")
display(ra_abstract_coherency.head(10))
print(f"\nLeast coherent abstracts (highest mean CV):")
display(ra_abstract_coherency.tail(10))

In [ ]:
# Visualization: Coherency Distribution
fig = px.histogram(
    prp_coherency[prp_coherency['cv'].notna()],
    x='cv',
    # nbins=30,
    # facet_col = 'PRP_Name',
    title='PRP Coherency Distribution (Coefficient of Variation)',
    labels={'cv': 'Coefficient of Variation', 'count': 'Number of Pairs'}
)
fig.add_vline(x=prp_coherency['cv'].mean(), line_dash="dash", line_color="red", 
              annotation_text=f"Mean CV: {prp_coherency['cv'].mean():.3f}")
fig.show()

fig = px.histogram(
    ra_coherency[ra_coherency['cv'].notna()],
    x='cv',
    nbins=30,
    title='RA Coherency Distribution (Coefficient of Variation)',
    labels={'cv': 'Coefficient of Variation', 'count': 'Number of Pairs'}
)
fig.add_vline(x=ra_coherency['cv'].mean(), line_dash="dash", line_color="red",
              annotation_text=f"Mean CV: {ra_coherency['cv'].mean():.3f}")
fig.show()

In [ ]:
prp_coherency

In [ ]:
# Abstract-level coherency scatter plots
fig = px.box(
    prp_coherency,
    x='Abstract_Index',
    y='cv',
    title='PRP Abstract-level Coherency (Lower CV = Higher Agreement)',
    points ='all',
)
fig.update_traces(boxmean=True)  # set to 'sd' to show mean + std
fig.show()

fig = px.box(
    ra_coherency,
    x='Abstract_Index',
    y='cv',
    title='RA Abstract-level Coherency (Lower CV = Higher Agreement)',
)
fig.update_traces(boxmean=True)  # set to 'sd' to show mean + std
fig.show()


In [ ]:
# Abstract-level coherency scatter plots
fig = px.scatter(
    prp_abstract_coherency,
    x='Abstract_Index',
    y='mean_cv',
    size='n_pairs',
    color='mean_cv',
    hover_data=['max_cv', 'std_cv'],
    title='PRP Abstract-level Coherency (Lower CV = Higher Agreement)',
    labels={'mean_cv': 'Mean CV (Coherency)', 'Abstract_Index': 'Abstract Index'},
    color_continuous_scale='RdYlGn_r'
)
fig.show()

fig = px.scatter(
    ra_abstract_coherency,
    x='Abstract_Index',
    y='mean_cv',
    size='n_pairs',
    color='mean_cv',
    hover_data=['max_cv', 'std_cv'],
    title='RA Abstract-level Coherency (Lower CV = Higher Agreement)',
    labels={'mean_cv': 'Mean CV (Coherency)', 'Abstract_Index': 'Abstract Index'},
    color_continuous_scale='RdYlGn_r'
)
fig.show()

In [ ]:
# Pairwise Model Coherency: Correlation between model pairs per abstract-PRP
# This shows which model pairs agree most on affinity levels

from itertools import combinations

# For PRP: compute correlation between each pair of models per abstract
prp_model_pairs = []
for abstract_idx in prp_df['Abstract_Index'].unique():
    for prp_name in prp_df[prp_df['Abstract_Index'] == abstract_idx]['PRP_Name'].unique():
        subset = prp_df[(prp_df['Abstract_Index'] == abstract_idx) & (prp_df['PRP_Name'] == prp_name)]
        
        if len(subset) >= 2:
            models = subset['Model_Slug'].unique()
            for model1, model2 in combinations(models, 2):
                val1 = subset[subset['Model_Slug'] == model1]['LLM_Affinity'].values
                val2 = subset[subset['Model_Slug'] == model2]['LLM_Affinity'].values
                
                if len(val1) > 0 and len(val2) > 0:
                    diff = abs(val1[0] - val2[0])
                    prp_model_pairs.append({
                        'Abstract_Index': abstract_idx,
                        'PRP_Name': prp_name,
                        'Model_1': min(model1, model2),
                        'Model_2': max(model1, model2),
                        'Affinity_Difference': diff
                    })

prp_pairwise = pd.DataFrame(prp_model_pairs)

if len(prp_pairwise) > 0:
    print("PRP Model Pair Agreement Analysis:")
    print("\nModel pairs with highest average disagreement (highest mean affinity difference):")
    pair_stats = prp_pairwise.groupby(['Model_1', 'Model_2'], as_index=False)['Affinity_Difference'].agg(
        mean_diff='mean', std_diff='std', n_pairs='count'
    ).sort_values('mean_diff', ascending=False)
    display(pair_stats.head(10))
    
    print("\nModel pairs with lowest average disagreement (highest mean agreement):")
    display(pair_stats.sort_values('mean_diff', ascending=True).head(10))

In [ ]:
# For RA: compute correlation between each pair of models per abstract
ra_model_pairs = []
for abstract_idx in ra_df['Abstract_Index'].unique():
    for ra_id in ra_df[ra_df['Abstract_Index'] == abstract_idx]['RA2025_ID'].unique():
        subset = ra_df[(ra_df['Abstract_Index'] == abstract_idx) & (ra_df['RA2025_ID'] == ra_id)]
        
        if len(subset) >= 2:
            models = subset['Model_Slug'].unique()
            for model1, model2 in combinations(models, 2):
                val1 = subset[subset['Model_Slug'] == model1]['LLM_Affinity'].values
                val2 = subset[subset['Model_Slug'] == model2]['LLM_Affinity'].values
                
                if len(val1) > 0 and len(val2) > 0:
                    diff = abs(val1[0] - val2[0])
                    ra_model_pairs.append({
                        'Abstract_Index': abstract_idx,
                        'RA2025_ID': ra_id,
                        'Model_1': min(model1, model2),
                        'Model_2': max(model1, model2),
                        'Affinity_Difference': diff
                    })

ra_pairwise = pd.DataFrame(ra_model_pairs)

if len(ra_pairwise) > 0:
    print("RA Model Pair Agreement Analysis:")
    print("\nModel pairs with highest average disagreement (highest mean affinity difference):")
    pair_stats_ra = ra_pairwise.groupby(['Model_1', 'Model_2'], as_index=False)['Affinity_Difference'].agg(
        mean_diff='mean', std_diff='std', n_pairs='count'
    ).sort_values('mean_diff', ascending=False)
    display(pair_stats_ra.head(10))
    
    print("\nModel pairs with lowest average disagreement (highest mean agreement):")
    display(pair_stats_ra.sort_values('mean_diff', ascending=True).head(10))

In [ ]:
# Model Agreement Heatmaps: Show which models have smallest disagreements
if len(prp_pairwise) > 0:
    pair_stats = prp_pairwise.groupby(['Model_1', 'Model_2'], as_index=False)['Affinity_Difference'].agg(
        mean_diff='mean', n_pairs='count'
    ).sort_values('mean_diff', ascending=True)
    
    # Create a symmetrical matrix for heatmap
    all_models = sorted(set(list(pair_stats['Model_1'].unique()) + list(pair_stats['Model_2'].unique())))
    model_agreement_matrix = pd.DataFrame(0.0, index=all_models, columns=all_models)
    
    for _, row in pair_stats.iterrows():
        model_agreement_matrix.loc[row['Model_1'], row['Model_2']] = row['mean_diff']
        model_agreement_matrix.loc[row['Model_2'], row['Model_1']] = row['mean_diff']
    
    fig = px.imshow(
        model_agreement_matrix,
        text_auto='.2f',
        color_continuous_scale='RdYlGn_r',
        title='PRP Model Pairwise Disagreement (Lower = Better Agreement)',
        labels={'color': 'Mean Affinity Difference'},
        aspect='auto'
    )
    fig.update_layout(xaxis_title='Model', yaxis_title='Model')
    fig.show()

if len(ra_pairwise) > 0:
    pair_stats_ra = ra_pairwise.groupby(['Model_1', 'Model_2'], as_index=False)['Affinity_Difference'].agg(
        mean_diff='mean', n_pairs='count'
    ).sort_values('mean_diff', ascending=True)
    
    all_models_ra = sorted(set(list(pair_stats_ra['Model_1'].unique()) + list(pair_stats_ra['Model_2'].unique())))
    model_agreement_matrix_ra = pd.DataFrame(0.0, index=all_models_ra, columns=all_models_ra)
    
    for _, row in pair_stats_ra.iterrows():
        model_agreement_matrix_ra.loc[row['Model_1'], row['Model_2']] = row['mean_diff']
        model_agreement_matrix_ra.loc[row['Model_2'], row['Model_1']] = row['mean_diff']
    
    fig = px.imshow(
        model_agreement_matrix_ra,
        text_auto='.2f',
        color_continuous_scale='RdYlGn_r',
        title='RA Model Pairwise Disagreement (Lower = Better Agreement)',
        labels={'color': 'Mean Affinity Difference'},
        aspect='auto'
    )
    fig.update_layout(xaxis_title='Model', yaxis_title='Model')
    fig.show()

## Coherency Key Insights

Summary of model agreement levels on affinity assignments.

In [ ]:
# Summary statistics on model coherency
print("=" * 70)
print("COHERENCY ANALYSIS SUMMARY")
print("=" * 70)

print("\n📊 PRP COHERENCY:")
print(f"  • Mean CV (lower = more agreement): {prp_coherency['cv'].mean():.3f}")
print(f"  • Median CV: {prp_coherency['cv'].median():.3f}")
print(f"  • Perfect agreement cases (CV=0): {(prp_coherency['cv'] == 0).sum()} / {len(prp_coherency)}")
print(f"  • High disagreement cases (CV>1): {(prp_coherency['cv'] > 1).sum()} / {len(prp_coherency)}")
print(f"  • Range of affinities (mean): {prp_coherency['range'].mean():.3f}")
print(f"  • Most coherent abstract: {prp_abstract_coherency.iloc[0]['Abstract_Index']} (mean CV: {prp_abstract_coherency.iloc[0]['mean_cv']:.3f})")
print(f"  • Least coherent abstract: {prp_abstract_coherency.iloc[-1]['Abstract_Index']} (mean CV: {prp_abstract_coherency.iloc[-1]['mean_cv']:.3f})")

print("\n📊 RA COHERENCY:")
print(f"  • Mean CV (lower = more agreement): {ra_coherency['cv'].mean():.3f}")
print(f"  • Median CV: {ra_coherency['cv'].median():.3f}")
print(f"  • Perfect agreement cases (CV=0): {(ra_coherency['cv'] == 0).sum()} / {len(ra_coherency)}")
print(f"  • High disagreement cases (CV>1): {(ra_coherency['cv'] > 1).sum()} / {len(ra_coherency)}")
print(f"  • Range of affinities (mean): {ra_coherency['range'].mean():.3f}")
print(f"  • Most coherent abstract: {ra_abstract_coherency.iloc[0]['Abstract_Index']} (mean CV: {ra_abstract_coherency.iloc[0]['mean_cv']:.3f})")
print(f"  • Least coherent abstract: {ra_abstract_coherency.iloc[-1]['Abstract_Index']} (mean CV: {ra_abstract_coherency.iloc[-1]['mean_cv']:.3f})")

if len(prp_pairwise) > 0:
    pair_stats = prp_pairwise.groupby(['Model_1', 'Model_2'], as_index=False)['Affinity_Difference'].mean()
    print("\n🔗 PRP MODEL PAIRS - BEST AGREEMENT (Lowest Mean Difference):")
    for i, (_, row) in enumerate(pair_stats.sort_values('Affinity_Difference').head(3).iterrows(), 1):
        print(f"  {i}. {row['Model_1']} ↔ {row['Model_2']}: {row['Affinity_Difference']:.3f}")

if len(ra_pairwise) > 0:
    pair_stats_ra = ra_pairwise.groupby(['Model_1', 'Model_2'], as_index=False)['Affinity_Difference'].mean()
    print("\n🔗 RA MODEL PAIRS - BEST AGREEMENT (Lowest Mean Difference):")
    for i, (_, row) in enumerate(pair_stats_ra.sort_values('Affinity_Difference').head(3).iterrows(), 1):
        print(f"  {i}. {row['Model_1']} ↔ {row['Model_2']}: {row['Affinity_Difference']:.3f}")

print("\n" + "=" * 70)

## Coherency by Fixed Affinity Levels (CV + MAD)

This section extends coherency analysis with fixed affinity bands and a robust companion metric:

- Low: 0-40
- Moderate: 41-70
- High: 71-100

Metrics per abstract-target unit across models:
- CV = std / mean (with epsilon)
- MAD = mean absolute deviation from cross-model mean

CV can be unstable when mean affinity is very low, so a low-mean caution flag is included.

In [ ]:
# PRP coherency by fixed levels
prp_unit_coherency_lvl = compute_unit_coherency(prp_df, target_col='PRP_Name')
prp_level_summary = summarize_coherency_by_level(prp_unit_coherency_lvl)

print('PRP level-wise coherency summary (CV + MAD):')
display(prp_level_summary)

print('\nMost incoherent PRP units by CV:')
display(
    prp_unit_coherency_lvl.sort_values('cv', ascending=False)[
        ['Abstract_Index', 'PRP_Name', 'mean_affinity', 'cv', 'mad_affinity', 'range', 'n_models', 'cv_low_mean_flag']
    ].head(15)
)

print('\nMost incoherent PRP units by MAD:')
display(
    prp_unit_coherency_lvl.sort_values('mad_affinity', ascending=False)[
        ['Abstract_Index', 'PRP_Name', 'mean_affinity', 'cv', 'mad_affinity', 'range', 'n_models', 'cv_low_mean_flag']
    ].head(15)
)

fig = px.box(
    prp_unit_coherency_lvl.dropna(subset=['Affinity_Level']),
    x='Affinity_Level',
    y='cv',
    color='Affinity_Level',
    category_orders={'Affinity_Level': ['Low', 'Moderate', 'High']},
    title='PRP Coherency by Affinity Level (CV)',
    points='all',
)
fig.update_layout(xaxis_title='Affinity level', yaxis_title='CV across models')
fig.show()

fig = px.box(
    prp_unit_coherency_lvl.dropna(subset=['Affinity_Level']),
    x='Affinity_Level',
    y='mad_affinity',
    color='Affinity_Level',
    category_orders={'Affinity_Level': ['Low', 'Moderate', 'High']},
    title='PRP Coherency by Affinity Level (MAD)',
    points='all',
)
fig.update_layout(xaxis_title='Affinity level', yaxis_title='MAD across models')
fig.show()

In [ ]:
# RA coherency by fixed levels
ra_unit_coherency_lvl = compute_unit_coherency(ra_df, target_col='RA2025_ID')
ra_level_summary = summarize_coherency_by_level(ra_unit_coherency_lvl)

print('RA level-wise coherency summary (CV + MAD):')
display(ra_level_summary)

print('\nMost incoherent RA units by CV:')
display(
    ra_unit_coherency_lvl.sort_values('cv', ascending=False)[
        ['Abstract_Index', 'RA2025_ID', 'mean_affinity', 'cv', 'mad_affinity', 'range', 'n_models', 'cv_low_mean_flag']
    ].head(15)
)

print('\nMost incoherent RA units by MAD:')
display(
    ra_unit_coherency_lvl.sort_values('mad_affinity', ascending=False)[
        ['Abstract_Index', 'RA2025_ID', 'mean_affinity', 'cv', 'mad_affinity', 'range', 'n_models', 'cv_low_mean_flag']
    ].head(15)
)

fig = px.box(
    ra_unit_coherency_lvl.dropna(subset=['Affinity_Level']),
    x='Affinity_Level',
    y='cv',
    color='Affinity_Level',
    category_orders={'Affinity_Level': ['Low', 'Moderate', 'High']},
    title='RA Coherency by Affinity Level (CV)'
)
fig.update_layout(xaxis_title='Affinity level', yaxis_title='CV across models')
fig.show()

fig = px.box(
    ra_unit_coherency_lvl.dropna(subset=['Affinity_Level']),
    x='Affinity_Level',
    y='mad_affinity',
    color='Affinity_Level',
    category_orders={'Affinity_Level': ['Low', 'Moderate', 'High']},
    title='RA Coherency by Affinity Level (MAD)'
)
fig.update_layout(xaxis_title='Affinity level', yaxis_title='MAD across models')
fig.show()

In [ ]:
# Pairwise model disagreement by level (PRP)
prp_pairwise_lvl, prp_pairwise_lvl_summary = compute_pairwise_disagreement_by_level(prp_df, target_col='PRP_Name')

if not prp_pairwise_lvl_summary.empty:
    print('PRP pairwise disagreement by level (highest mean difference):')
    display(prp_pairwise_lvl_summary.head(20))

    for level in ['Low', 'Moderate', 'High']:
        level_df = prp_pairwise_lvl_summary[prp_pairwise_lvl_summary['Affinity_Level'] == level]
        if level_df.empty:
            continue

        models = sorted(set(level_df['Model_1']).union(level_df['Model_2']))
        mat = pd.DataFrame(np.nan, index=models, columns=models)
        np.fill_diagonal(mat.values, 0.0)

        for _, row in level_df.iterrows():
            mat.loc[row['Model_1'], row['Model_2']] = row['mean_diff']
            mat.loc[row['Model_2'], row['Model_1']] = row['mean_diff']

        fig = px.imshow(
            mat,
            text_auto='.2f',
            color_continuous_scale='RdYlGn_r',
            title=f'PRP Pairwise Disagreement ({level})',
            labels={'color': 'Mean absolute difference'},
            aspect='auto'
        )
        fig.update_layout(xaxis_title='Model', yaxis_title='Model')
        fig.show()
else:
    print('No PRP pairwise level summary available.')

In [ ]:
# Pairwise model disagreement by level (RA)
ra_pairwise_lvl, ra_pairwise_lvl_summary = compute_pairwise_disagreement_by_level(ra_df, target_col='RA2025_ID')

if not ra_pairwise_lvl_summary.empty:
    print('RA pairwise disagreement by level (highest mean difference):')
    display(ra_pairwise_lvl_summary.head(20))

    for level in ['Low', 'Moderate', 'High']:
        level_df = ra_pairwise_lvl_summary[ra_pairwise_lvl_summary['Affinity_Level'] == level]
        if level_df.empty:
            continue

        models = sorted(set(level_df['Model_1']).union(level_df['Model_2']))
        mat = pd.DataFrame(np.nan, index=models, columns=models)
        np.fill_diagonal(mat.values, 0.0)

        for _, row in level_df.iterrows():
            mat.loc[row['Model_1'], row['Model_2']] = row['mean_diff']
            mat.loc[row['Model_2'], row['Model_1']] = row['mean_diff']

        fig = px.imshow(
            mat,
            text_auto='.2f',
            color_continuous_scale='RdYlGn_r',
            title=f'RA Pairwise Disagreement ({level})',
            labels={'color': 'Mean absolute difference'},
            aspect='auto'
        )
        fig.update_layout(xaxis_title='Model', yaxis_title='Model')
        fig.show()
else:
    print('No RA pairwise level summary available.')

In [ ]:
# Compact interpretation table
interpretation_summary = pd.DataFrame(
    [
        {
            'Dataset': 'PRP',
            'Mean_CV': prp_unit_coherency_lvl['cv'].mean(),
            'Median_CV': prp_unit_coherency_lvl['cv'].median(),
            'Mean_MAD': prp_unit_coherency_lvl['mad_affinity'].mean(),
            'Median_MAD': prp_unit_coherency_lvl['mad_affinity'].median(),
            'Pct_Low_Mean_Flag': 100.0 * prp_unit_coherency_lvl['cv_low_mean_flag'].mean(),
            'N_Units': len(prp_unit_coherency_lvl),
        },
        {
            'Dataset': 'RA',
            'Mean_CV': ra_unit_coherency_lvl['cv'].mean(),
            'Median_CV': ra_unit_coherency_lvl['cv'].median(),
            'Mean_MAD': ra_unit_coherency_lvl['mad_affinity'].mean(),
            'Median_MAD': ra_unit_coherency_lvl['mad_affinity'].median(),
            'Pct_Low_Mean_Flag': 100.0 * ra_unit_coherency_lvl['cv_low_mean_flag'].mean(),
            'N_Units': len(ra_unit_coherency_lvl),
        },
    ]
)

print('Global coherency summary (CV + MAD):')
display(interpretation_summary)

print('Interpretation notes:')
print('- Low CV + Low MAD: strong cross-model agreement')
print('- High CV + Low mean affinity: inspect MAD before concluding disagreement')
print('- High CV + High MAD: substantive disagreement across models')